In [ ]:
!pip install google-play-scraper pandas nltk Sastrawi scikit-learn matplotlib seaborn



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1. Scraping Review TIX ID App

In [ ]:
from google_play_scraper import reviews_all, Sort
import pandas as pd

tix_reviews = reviews_all(
    'id.tix.android',
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

df = pd.DataFrame(tix_reviews)
df = df[['content', 'score', 'at']]
df = df.dropna(subset=['content'])

df = df[df['content'].str.strip() != '']

print(f'Total Review: {len(df)}')
df.head()

Total Review: 96278


,content,score,at
0,tes cek dulu,4,2026-04-06 15:01:24
1,"kota bitung gak ada,yg ada hanya manado",5,2026-04-06 12:41:05
2,dear developer aplikasi ini tolong updatean ba...,5,2026-04-06 11:10:27
3,sangat baik,5,2026-04-05 06:22:05
4,sangat bagus,5,2026-04-03 12:55:28


2. Label Sentiment

In [ ]:
def label_sentiment(score: int) -> str:
    if score >= 4:
        return 'Positive'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Negative'

df['sentiment'] = df['score'].apply(label_sentiment)

3. Regex (Text Cleaning)

In [ ]:
import re

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)  # huruf berulang
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_content'] = df['content'].apply(clean_text)


4. Stopword Removal

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop = stopwords.words('indonesian')

df['no_stopwords'] = df['clean_content'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop])
)

AttributeError: partially initialized module 'nltk' has no attribute 'data' (most likely due to a circular import)

5. Stemming

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

df['stemmed'] = df['no_stopwords'].apply(stemmer.stem)

6. Bag of Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# ambil 50 data
df_sample = df.sample(50, random_state=42)

bow_vectorizer = CountVectorizer(max_features=20)

X_bow = bow_vectorizer.fit_transform(df_sample['stemmed'])

bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

print("=== BoW Matrix ===")
print(bow_df.head())

print("\nShape:", bow_df.shape)

6a. Split Data

In [ ]:
df_pos = df[df['sentiment'] == 'Positive'].sample(50, random_state=42)
df_neg = df[df['sentiment'] == 'Negative'].sample(50, random_state=42)

6b. BoW Positive & Negative

In [ ]:
vectorizer_pos = CountVectorizer(max_features=20)
X_pos = vectorizer_pos.fit_transform(df_pos['stemmed'])

bow_pos = pd.DataFrame(
    X_pos.toarray(),
    columns=vectorizer_pos.get_feature_names_out()
)

print("=== BoW Positive ===")
print(bow_pos.head())

vectorizer_neg = CountVectorizer(max_features=20)
X_neg = vectorizer_neg.fit_transform(df_neg['stemmed'])

bow_neg = pd.DataFrame(
    X_neg.toarray(),
    columns=vectorizer_neg.get_feature_names_out()
)

print("\n=== BoW Negative ===")
print(bow_neg.head())

7. Analisis BoW

In [ ]:
word_freq = bow_df.sum().sort_values(ascending=False)

print("\n=== Top Words ===")
print(word_freq.head(10))

pos_words = bow_pos.sum().sort_values(ascending=False)
neg_words = bow_neg.sum().sort_values(ascending=False)

print("\n=== Top Positive Words ===")
print(pos_words.head(10))

print("\n=== Top Negative Words ===")
print(neg_words.head(10))

8. Regex Insight

In [ ]:
issue_patterns = {
    'login_issue': r'login|masuk|signin',
    'payment_issue': r'bayar|pembayaran|gagal bayar',
    'app_crash': r'error|crash|force close|bug',
    'slow_app': r'lambat|lemot|loading lama',
    'ticket_issue': r'tiket|kursi|booking'
}

def detect_issues(text: str):
    found = []
    for issue, pattern in issue_patterns.items():
        if re.search(pattern, text):
            found.append(issue)
    return found

df['issues'] = df['clean_content'].apply(detect_issues)

9. Analisis Regex

In [ ]:
from collections import Counter

all_issues = sum(df['issues'], [])
issue_counts = Counter(all_issues)

print("\n=== Issue Counts ===")
print(issue_counts)

10. Visualisasi

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(issue_counts.keys(), issue_counts.values())
plt.title("Distribusi Issue dari Review (Regex)")
plt.xticks(rotation=45)
plt.show()